# Run02

In [1]:
VERSION = 3

import pandas as pd
import numpy as np
from catboost import CatBoostRegressor, Pool


from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge, LassoCV, ElasticNetCV
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, GroupShuffleSplit, GroupKFold
from scipy.stats import pearsonr, spearmanr


In [2]:
df = pd.read_csv(r"C:\Users\Pratham Shah\Downloads\Tubingen\Tabular_Models_Dataset\Updated01\community_index_v8_backup.csv")
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

Loaded: 30,000 rows × 168 columns


In [3]:
# feature selection controlled by version

r0_cols     = [f"R0_c{k}"     for k in range(1, 21)]
active_cols = [f"active_c{k}" for k in range(1, 21)]

if VERSION == 1:
    species_cols = [f"Species_{i:04d}" for i in range(2, 102)]
    feature_cols = species_cols + r0_cols + active_cols + ['community_size', 'diversity_level']
    version_label = "Full (species + nutrients + size)"

elif VERSION == 2:
    feature_cols = r0_cols + active_cols + ['community_size', 'diversity_level']
    version_label = "Nutrients + size (no species)"

elif VERSION == 3:
    feature_cols = r0_cols + active_cols
    version_label = "Pure nutrients only (no species, no size)"

else:
    raise ValueError("VERSION must be 1, 2, or 3")

print(f"\nVersion: {version_label}")
print(f"Features: {len(feature_cols)}")

missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise KeyError(f"{len(missing)} columns not found. First 3: {missing[:3]}\n"
                   f"Make sure you are loading community_index_v8_backup.csv (30,000 rows), "
                   f"not community_dataset_v8_backup.csv (792,000 rows).")

X = df[feature_cols]
y = df['community_suppression']
print(f"X: {X.shape}   y range: [{y.min():.3f}, {y.max():.3f}]")


Version: Pure nutrients only (no species, no size)
Features: 40
X: (30000, 40)   y range: [0.000, 0.999]


In [4]:
# train test split

strat_key = df['community_size'].astype(str) + "_" + df['diversity_level'].astype(str)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=strat_key
)
print(f"\nTrain: {X_train.shape[0]:,}   Test: {X_test.shape[0]:,}")

groups = df['broth_id'].values # used later for groupkfold

# Wrap train and test in CatBoost Pool objects.
# Pool(data, label): packages features and target together.
# CatBoost uses Pool internally for all operations; passing Pools directly
# avoids repeated data conversion overhead during training.
# cat_features: list of column indices that are categorical.
#   All our features are numeric so this is an empty list.

train_pool = Pool(X_train, label=y_train, cat_features=[])
test_pool  = Pool(X_test,  label=y_test,  cat_features=[])


Train: 24,000   Test: 6,000


In [5]:
# catboost initial fir

# note about l2_leaf_reg:
#  L2 regularisation on leaf values. Equivalent to
#  reg_lambda in XGBoost. Default is 3.0 (stronger than
#  XGBoost's default of 1.0)

model_init = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=3.0,
    subsample=0.8,
    colsample_bylevel=0.8,
    early_stopping_rounds=30,
    use_best_model=True,
    eval_metric='RMSE',
    random_seed=42,
    thread_count=-1,
    verbose=0
)

# Pass Pool objects directly 
# eval_set here is the test Pool for early stopping monitoring.
model_init.fit(train_pool, eval_set=test_pool)

yp_init  = model_init.predict(X_test)
r2_init  = r2_score(y_test, yp_init)
mae_init = mean_absolute_error(y_test, yp_init)
print(f"CatBoost initial:  R²={r2_init:.4f}  MAE={mae_init:.4f}  "
      f"best_iter={model_init.best_iteration_}")

CatBoost initial:  R²=0.6812  MAE=0.1284  best_iter=208


In [6]:
# hyperparameter tuning

param_grid = {                            # using best hyperparams from first run
    'depth':            [3],
    # depth is CatBoost's primary complexity control.
    # depth=4 → 16 leaves, depth=8 → 256 leaves.

    'learning_rate':    [0.05],

    'l2_leaf_reg':      [3.0],
    # CatBoost-specific. Higher = more regularisation.
    # Default 3.0 is already stronger than XGBoost/LightGBM defaults.
    # Try 10.0 for VERSION 3 (pure nutrient) where overfitting is more likely.

    'colsample_bylevel':[1.0],
}


cv = KFold(n_splits=3, shuffle=True, random_state=42)

base_est = CatBoostRegressor(
    iterations=300,
    subsample=0.8,
    early_stopping_rounds=None,   # disabled inside GridSearchCV
    use_best_model=False,         # disabled inside GridSearchCV
    eval_metric='RMSE',
    random_seed=42,
    thread_count=-1,
    verbose=0
)

grid_search = GridSearchCV(
    estimator=base_est,
    param_grid=param_grid,
    cv=cv,
    scoring='neg_mean_squared_error',
    n_jobs=1,
    # n_jobs=1 here (not -1): CatBoost already uses all threads internally
    # via thread_count=-1. Running outer parallelism on top causes resource
    # contention and can crash. Keep outer n_jobs=1 for CatBoost.
    verbose=1,
    refit=True
)
grid_search.fit(X_train, y_train)

print(f"\nBest params: {grid_search.best_params_}")
print(f"Best CV RMSE: {np.sqrt(-grid_search.best_score_):.4f}")

Fitting 3 folds for each of 1 candidates, totalling 3 fits

Best params: {'colsample_bylevel': 1.0, 'depth': 3, 'l2_leaf_reg': 3.0, 'learning_rate': 0.05}
Best CV RMSE: 0.1831


In [8]:
# final model

model_final = CatBoostRegressor(
    **grid_search.best_params_,
    iterations=1000,
    subsample=0.8,
    early_stopping_rounds=40,
    use_best_model=True,
    eval_metric='RMSE',
    random_seed=42,
    thread_count=-1,
    verbose=50
)

model_final.fit(train_pool, eval_set=[train_pool, test_pool])

evals    = model_final.get_evals_result()
tr_rmse  = evals['learn']['RMSE']
val_rmse = evals['validation_0']['RMSE']
rounds   = range(1, len(tr_rmse) + 1)
best_it  = model_final.best_iteration_

fig_tc, ax_tc = plt.subplots(figsize=(9, 4))
ax_tc.plot(rounds, tr_rmse,  color='steelblue',  lw=1.5, label='Train RMSE')
ax_tc.plot(rounds, val_rmse, color='darkorange', lw=1.5, label='Validation RMSE')
ax_tc.axvline(best_it + 1, color='red', linestyle='--', lw=1.2,
              label=f'Best iteration = {best_it + 1}')
ax_tc.set_xlabel('Boosting round')
ax_tc.set_ylabel('RMSE')
ax_tc.set_title(f'CatBoost Training Curve  --  best iter={best_it + 1},  val RMSE={val_rmse[best_it]:.4f}')
ax_tc.legend(fontsize=9)
ax_tc.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('catboost_training_curve_0103.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: catboost_training_curve_0103.png")

yp_final   = model_final.predict(X_test)
r2_final   = r2_score(y_test, yp_final)
mae_final  = mean_absolute_error(y_test, yp_final)
rmse_final = np.sqrt(mean_squared_error(y_test, yp_final))

print(f"\nCatBoost final:  R²={r2_final:.4f}  MAE={mae_final:.4f}  RMSE={rmse_final:.4f}")
print(f"Trees used: {model_final.best_iteration_}")

0:	learn: 0.3110862	test: 0.3110862	test1: 0.3147316	best: 0.3147316 (0)	total: 10.1ms	remaining: 10.1s
50:	learn: 0.1899008	test: 0.1899008	test1: 0.1902761	best: 0.1902761 (50)	total: 172ms	remaining: 3.19s
100:	learn: 0.1836853	test: 0.1836853	test1: 0.1837290	best: 0.1837290 (100)	total: 280ms	remaining: 2.49s
150:	learn: 0.1830568	test: 0.1830568	test1: 0.1830256	best: 0.1830256 (150)	total: 396ms	remaining: 2.23s
200:	learn: 0.1829591	test: 0.1829591	test1: 0.1829106	best: 0.1829099 (196)	total: 509ms	remaining: 2.02s
250:	learn: 0.1829298	test: 0.1829298	test1: 0.1828911	best: 0.1828911 (250)	total: 616ms	remaining: 1.84s
300:	learn: 0.1829238	test: 0.1829238	test1: 0.1828889	best: 0.1828888 (271)	total: 734ms	remaining: 1.7s
350:	learn: 0.1829229	test: 0.1829229	test1: 0.1828881	best: 0.1828872 (338)	total: 845ms	remaining: 1.56s
Stopped by overfitting detector  (40 iterations wait)

bestTest = 0.1828871879
bestIteration = 338

Shrink model to first 339 iterations.
Saved: catbo

In [9]:
# groupkfold

gkf          = GroupKFold(n_splits=5)
fold_r2, fold_mae, fold_rmse = [], [], []

print("\nGroupKFold evaluation (held-out broths):")

for fold_num, (tr_idx, te_idx) in enumerate(
        gkf.split(X, y, groups=groups), start=1):

    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_te, y_te = X.iloc[te_idx], y.iloc[te_idx]

    held_broths = sorted(set(df.iloc[te_idx]['broth_id']))
    held_divs   = sorted(df.iloc[te_idx]['diversity_level'].unique())
    print(f"\nFold {fold_num}: broths={held_broths}  divs={held_divs}  n_test={len(X_te):,}")

    tr_pool = Pool(X_tr, label=y_tr, cat_features=[])
    te_pool = Pool(X_te, label=y_te, cat_features=[])

    m_fold = CatBoostRegressor(
        **grid_search.best_params_,
        iterations=1000, subsample=0.8,
        early_stopping_rounds=40, use_best_model=True,
        eval_metric='RMSE', random_seed=42, thread_count=-1, verbose=0
    )
    m_fold.fit(tr_pool, eval_set=te_pool)

    yp_fold = m_fold.predict(X_te)
    r2_f    = r2_score(y_te, yp_fold)
    mae_f   = mean_absolute_error(y_te, yp_fold)
    rmse_f  = np.sqrt(mean_squared_error(y_te, yp_fold))

    fold_r2.append(r2_f); fold_mae.append(mae_f); fold_rmse.append(rmse_f)
    print(f"  R²={r2_f:.4f}  MAE={mae_f:.4f}  RMSE={rmse_f:.4f}")

print(f"\nGroupKFold summary across {gkf.n_splits} folds:")
print(f"  R²:   mean={np.mean(fold_r2):.4f}  std={np.std(fold_r2):.4f}")
print(f"  MAE:  mean={np.mean(fold_mae):.4f}  std={np.std(fold_mae):.4f}")
print(f"  RMSE: mean={np.mean(fold_rmse):.4f}  std={np.std(fold_rmse):.4f}")
print(f"\nCompare to stratified-split R²={r2_final:.4f}")
print(f"Drop: {r2_final - np.mean(fold_r2):.4f}  (how much was broth memorisation)")


GroupKFold evaluation (held-out broths):

Fold 1: broths=['Broth_05', 'Broth_10', 'Broth_15', 'Broth_20']  divs=[np.int64(2), np.int64(4), np.int64(8), np.int64(16)]  n_test=6,000
  R²=0.0368  MAE=0.1765  RMSE=0.2208

Fold 2: broths=['Broth_04', 'Broth_09', 'Broth_14', 'Broth_19']  divs=[np.int64(1), np.int64(4), np.int64(8), np.int64(16)]  n_test=6,000
  R²=0.3100  MAE=0.1915  RMSE=0.2189

Fold 3: broths=['Broth_03', 'Broth_08', 'Broth_13', 'Broth_18']  divs=[np.int64(1), np.int64(2), np.int64(8), np.int64(16)]  n_test=6,000
  R²=-2.0560  MAE=0.4711  RMSE=0.5309

Fold 4: broths=['Broth_02', 'Broth_07', 'Broth_12', 'Broth_17']  divs=[np.int64(1), np.int64(2), np.int64(4), np.int64(16)]  n_test=6,000
  R²=0.5834  MAE=0.1376  RMSE=0.1740

Fold 5: broths=['Broth_01', 'Broth_06', 'Broth_11', 'Broth_16']  divs=[np.int64(1), np.int64(2), np.int64(4), np.int64(8)]  n_test=6,000
  R²=-0.0517  MAE=0.1880  RMSE=0.2381

GroupKFold summary across 5 folds:
  R²:   mean=-0.2355  std=0.9370
  MAE:  

In [10]:
print("Performance by diversity level and community size")

results_df = pd.DataFrame({
    'y_true':          y_test.values,
    'y_pred':          yp_final,
    'diversity_level': df.loc[X_test.index, 'diversity_level'].values,
    'community_size':  df.loc[X_test.index, 'community_size'].values,
})

def condition_metrics(y_true_arr, y_pred_arr):
    y_t      = np.asarray(y_true_arr)
    y_p      = np.asarray(y_pred_arr)
    r2       = r2_score(y_t, y_p)
    mae      = mean_absolute_error(y_t, y_p)
    rmse     = np.sqrt(mean_squared_error(y_t, y_p))
    pr, _    = pearsonr(y_t, y_p)
    sr, _    = spearmanr(y_t, y_p)
    bias     = float(np.mean(y_p - y_t))
    std_true = float(np.std(y_t))
    return {'n': len(y_t), 'r2': r2, 'mae': mae, 'rmse': rmse,
            'pearson_r': pr, 'spearman_r': sr,
            'bias': bias, 'std_true': std_true}

hdr = f"{'':6} {'R²':>7} {'MAE':>7} {'RMSE':>7} {'Pearson':>8} {'Spearman':>9} {'Bias':>8} {'std(y)':>7} {'N':>6}"
print("\nBy diversity level:")
print(hdr)
print("-" * len(hdr))
for d, grp in results_df.groupby('diversity_level'):
    m    = condition_metrics(grp['y_true'], grp['y_pred'])
    flag = "  *" if m['std_true'] < 0.08 else "   "
    print(f"D={d:<3}  {m['r2']:>7.4f} {m['mae']:>7.4f} {m['rmse']:>7.4f} "
          f"{m['pearson_r']:>8.4f} {m['spearman_r']:>9.4f} "
          f"{m['bias']:>8.4f} {m['std_true']:>7.4f} {m['n']:>6,}{flag}")
print("  * std(y) < 0.08: R² is unreliable here. Read MAE and Spearman r instead.")

print("\nBy community size:")
print(hdr)
print("-" * len(hdr))
for n, grp in results_df.groupby('community_size'):
    m = condition_metrics(grp['y_true'], grp['y_pred'])
    print(f"N={n:<3}  {m['r2']:>7.4f} {m['mae']:>7.4f} {m['rmse']:>7.4f} "
          f"{m['pearson_r']:>8.4f} {m['spearman_r']:>9.4f} "
          f"{m['bias']:>8.4f} {m['std_true']:>7.4f} {m['n']:>6,}")

Performance by diversity level and community size

By diversity level:
            R²     MAE    RMSE  Pearson  Spearman     Bias  std(y)      N
-------------------------------------------------------------------------
D=1     0.9037  0.0701  0.1314   0.9507    0.8661  -0.0045  0.4234  1,200   
D=2     0.2912  0.2052  0.2627   0.5398    0.5913   0.0041  0.3120  1,200   
D=4     0.2527  0.1456  0.1970   0.5052    0.5567   0.0092  0.2279  1,200   
D=8     0.8152  0.0951  0.1411   0.9030    0.8376   0.0023  0.3282  1,200   
D=16    0.2542  0.1256  0.1491   0.5054    0.4523  -0.0011  0.1727  1,200   
  * std(y) < 0.08: R² is unreliable here. Read MAE and Spearman r instead.

By community size:
            R²     MAE    RMSE  Pearson  Spearman     Bias  std(y)      N
-------------------------------------------------------------------------
N=12    0.4314  0.1826  0.2573   0.7671    0.7808   0.1352  0.3412  1,200
N=16    0.6755  0.1298  0.1879   0.8402    0.8395   0.0561  0.3298  1,200
N=24 

In [11]:
# shap analysis

print('shap analysis')

# CatBoost has its own built-in SHAP implementation accessible via
# get_feature_importance(type='ShapValues'), which is fast and exact.
# However, shap.TreeExplainer also supports CatBoost 
# we use it here to keep the interface identical across all three
# model scripts so results are directly comparable.
explainer   = shap.TreeExplainer(model_final)
rng         = np.random.default_rng(42)
idx         = rng.choice(len(X_test), size=2000, replace=False)
shap_values = explainer.shap_values(X_test.iloc[idx])
mean_shap   = pd.Series(np.abs(shap_values).mean(0), index=feature_cols).sort_values(ascending=False)

print(f"\nTop 15 features by mean |SHAP|:")
print(mean_shap.head(15).round(5).to_string())

shap analysis

Top 15 features by mean |SHAP|:
R0_c1         0.07628
R0_c8         0.04980
active_c8     0.04534
R0_c9         0.02477
R0_c19        0.02244
R0_c20        0.01809
active_c20    0.01629
R0_c3         0.01081
R0_c15        0.00865
R0_c18        0.00794
R0_c17        0.00789
R0_c13        0.00774
R0_c16        0.00667
R0_c6         0.00611
R0_c12        0.00574


In [12]:
# figures

from matplotlib.patches import Patch

output_name = f"catboost_v{VERSION}_results_01.png"

fig = plt.figure(figsize=(18, 14), dpi=120)
fig.suptitle(
    f'CatBoost — {version_label} — MiCRM v8\n'
    f'R²={r2_final:.3f}  MAE={mae_final:.3f}  RMSE={rmse_final:.3f}',
    fontsize=12, fontweight='bold', y=0.99
)

ax1 = fig.add_subplot(2, 3, 1)
for d, col in zip([1,2,4,8,16], ['#e41a1c','#ff7f00','#4daf4a','#377eb8','#984ea3']):
    mask = results_df['diversity_level'].values == d
    ax1.scatter(y_test.values[mask], yp_final[mask], alpha=0.15, s=2, c=col, label=f'D={d}')
ax1.plot([0,1],[0,1],'r--',lw=1.5)
ax1.set_xlabel('True suppression'); ax1.set_ylabel('Predicted')
ax1.set_title(f'Predicted vs Actual  R²={r2_final:.3f}')
ax1.legend(fontsize=7, markerscale=3)

ax2 = fig.add_subplot(2, 3, 2)
resid = y_test.values - yp_final
for n, col in zip([12,16,24,32,48], ['#1b9e77','#d95f02','#7570b3','#e7298a','#66a61e']):
    mask = results_df['community_size'].values == n
    ax2.scatter(yp_final[mask], resid[mask], alpha=0.15, s=2, c=col, label=f'N={n}')
ax2.axhline(0, color='r', linestyle='--', lw=1.5)
p5, p95 = np.percentile(resid, [5, 95])
ax2.axhline(p5,  color='grey', linestyle=':', lw=1, label=f'5th: {p5:.3f}')
ax2.axhline(p95, color='grey', linestyle=':', lw=1, label=f'95th: {p95:.3f}')
ax2.set_xlabel('Predicted'); ax2.set_ylabel('Residual')
ax2.set_title('Residuals by Community Size')
ax2.legend(fontsize=7, markerscale=3)

ax3 = fig.add_subplot(2, 3, 3)
top_n    = min(25, len(feature_cols))
top_shap = mean_shap.head(top_n)
def feat_col(f):
    if 'Species' in f: return '#2196F3'
    if 'R0'      in f: return '#4CAF50'
    if 'active'  in f: return '#FF9800'
    return '#9C27B0'
cols = [feat_col(f) for f in top_shap.index]
ax3.barh(range(top_n), top_shap.values[::-1], color=cols[::-1])
ax3.set_yticks(range(top_n))
ax3.set_yticklabels(top_shap.index[::-1], fontsize=7)
ax3.set_xlabel('Mean |SHAP|'); ax3.set_title(f'Top {top_n} Features (SHAP)')
ax3.legend(handles=[
    Patch(fc='#2196F3', label='Species'),
    Patch(fc='#4CAF50', label='R0'),
    Patch(fc='#FF9800', label='Active'),
    Patch(fc='#9C27B0', label='Structural'),
], loc='lower right', fontsize=7)

ax4 = fig.add_subplot(2, 3, 4)
divs   = sorted(results_df['diversity_level'].unique())
r2_div = [r2_score(results_df[results_df['diversity_level']==d]['y_true'],
                   results_df[results_df['diversity_level']==d]['y_pred']) for d in divs]
ax4.bar([str(d) for d in divs], r2_div, color='steelblue', edgecolor='navy')
ax4.axhline(r2_final, color='red', linestyle='--', lw=1.5, label=f'Overall R²={r2_final:.3f}')
ax4.set_xlabel('Diversity level'); ax4.set_ylabel('R²')
ax4.set_title('R² by Diversity Level'); ax4.set_ylim(0, 1.05); ax4.legend(fontsize=8)

ax5 = fig.add_subplot(2, 3, 5)
sizes  = sorted(results_df['community_size'].unique())
r2_sz  = [r2_score(results_df[results_df['community_size']==n]['y_true'],
                   results_df[results_df['community_size']==n]['y_pred']) for n in sizes]
ax5.bar([str(n) for n in sizes], r2_sz, color='darkorange', edgecolor='saddlebrown')
ax5.axhline(r2_final, color='red', linestyle='--', lw=1.5, label=f'Overall R²={r2_final:.3f}')
ax5.set_xlabel('Community size N'); ax5.set_ylabel('R²')
ax5.set_title('R² by Community Size'); ax5.set_ylim(0, 1.05); ax5.legend(fontsize=8)

ax6 = fig.add_subplot(2, 3, 6)
groups, values, colours = [], [], []
sp_s = mean_shap[[f for f in feature_cols if 'Species' in f]].sum()
if sp_s > 0:
    groups.append('Species'); values.append(sp_s); colours.append('#2196F3')
r0_s = mean_shap[r0_cols].sum()
groups.append('R0'); values.append(r0_s); colours.append('#4CAF50')
ac_s = mean_shap[active_cols].sum()
groups.append('Active'); values.append(ac_s); colours.append('#FF9800')
st_s = mean_shap[[f for f in feature_cols if f in ['community_size','diversity_level']]].sum()
if st_s > 0:
    groups.append('Structural'); values.append(st_s); colours.append('#9C27B0')
ax6.pie(values, labels=groups, colors=colours,
        autopct='%.1f%%', startangle=90, textprops={'fontsize': 10})
ax6.set_title('SHAP by Feature Group')

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(output_name, dpi=150, bbox_inches='tight')
plt.close()
print(f"\nPlot saved: {output_name}")


Plot saved: catboost_v3_results_01.png


In [ ]:
# save outputs 

model_save = f"catboost_v{VERSION}_model.cbm"
pred_save  = f"catboost_v{VERSION}_predictions.csv"

# CatBoost saves in its own .cbm (CatBoost Model) binary format.
# Reload with: m = CatBoostRegressor(); m.load_model('catboost_v1_model.cbm')
model_final.save_model(model_save)

pd.DataFrame({
    'y_true':          y_test.values,
    'y_pred':          yp_final,
    'residual':        y_test.values - yp_final,
    'diversity_level': results_df['diversity_level'].values,
    'community_size':  results_df['community_size'].values,
}).to_csv(pred_save, index=False)

print(f"Model saved:       {model_save}")
print(f"Predictions saved: {pred_save}")

In [13]:
# summary

print(f"\n{'='*60}")
print(f"SUMMARY — CatBoost VERSION {VERSION}: {version_label}")
print(f"{'='*60}")
print(f"Features:  {len(feature_cols)}")
print(f"R²:        {r2_final:.4f}")
print(f"MAE:       {mae_final:.4f}")
print(f"RMSE:      {rmse_final:.4f}")
print(f"Trees:     {model_final.best_iteration_}")
print(f"\nTop 5 features:")
for feat, val in mean_shap.head(5).items():
    print(f"  {feat:<25} {val:.5f}")


SUMMARY — CatBoost VERSION 3: Pure nutrients only (no species, no size)
Features:  40
R²:        0.6812
MAE:       0.1283
RMSE:      0.1829
Trees:     338

Top 5 features:
  R0_c1                     0.07628
  R0_c8                     0.04980
  active_c8                 0.04534
  R0_c9                     0.02477
  R0_c19                    0.02244
